In [1]:
import pandas as pd
import numpy as np

from datetime import datetime, timedelta
from tqdm import tqdm

save_path = "C:\\Users\\Happy\\Desktop\\맹박사님\\data\\"

In [2]:
# yes traider 파일
KRW_USDT2 = pd.read_csv(r"C:\Users\Happy\Desktop\맹박사님\data\daydata_2.csv", encoding = "cp949",names = ['날짜',"오픈","고가","저가","종가"])

def Pretreatment_by_yes_trader(df): #데이터 가이드에서 다운한 데이터 전처리
    df["날짜"] = [df["날짜"][i][0:4]+"-"+df["날짜"][i][5:7]+"-"+df["날짜"][i][8:10] for i in range(len(df["날짜"]))]
    df['날짜'] = pd.to_datetime(df['날짜'], format='%Y-%m-%d', errors='raise')
    
    # 예스트레이더는 장이 열려있으면 데이터를 이상하게 주기 떄문에 중복제거, 인덱스 초기화 를 하였음
    df = df.sort_values(['날짜'],ascending= True).drop_duplicates('날짜').reset_index(drop = True).rename(columns={'날짜':'time'})
    
    return df

df = Pretreatment_by_yes_trader(KRW_USDT2)

In [3]:
df = df[["time","종가","오픈","고가","저가"]]
df["day_of_year"] = df.time.dt.day_of_year
df["weekofyear"] = df.time.dt.weekofyear
df["year"] = df.time.dt.year
df["day_of_week"] = df.time.dt.day_of_week
df["year_of_month"]  = df.time.dt.month

C:\Users\Happy\AppData\Local\Temp\ipykernel_14272\1363962668.py:3: FutureWarning: Series.dt.weekofyear and Series.dt.week have been deprecated. Please use Series.dt.isocalendar().week instead.
  df["weekofyear"] = df.time.dt.weekofyear


In [4]:
df.loc[(df.year_of_month == 12)&(df.weekofyear == 1)]

,time,종가,오픈,고가,저가,day_of_year,weekofyear,year,day_of_week,year_of_month
68,2019-12-30,286.8,287.4,288.15,286.45,364,1,2019,0,12


In [5]:
df.loc[(df.year_of_month == 12)&(df.weekofyear == 1),"weekofyear"] = 53

def weekofyear_(date):
    if ((date.month == 12) & (date.weekofyear== 1)):
         return 53
    else:
        return date.weekofyear



In [6]:
df.to_csv("future_rowdata.csv", encoding = "cp949")
df = df.drop_duplicates()

In [7]:
df.columns

Index(['time', '종가', '오픈', '고가', '저가', 'day_of_year', 'weekofyear', 'year',
       'day_of_week', 'year_of_month'],
      dtype='object')

In [8]:
def out_put(df, start_dayofweek=4, end_dayofweek=3):
    info = {"time": 0, "오픈": 0, "저가": 0, "고가": 0, "종가": None, "진입시점": 0}
    info_dataFrame = pd.DataFrame(info, index=[0])

    df1 = df[["year", "weekofyear"]].drop_duplicates()

    for i in tqdm(range(len(df1))):
#     for i in range(len(df1)):

        df_strat_day = df.loc[(df1.iloc[i, :].year == df.year) & (df1.iloc[i, :].weekofyear == df.weekofyear)].time.iloc[0]  # 시작일
        while True:
            new_df_1 = df.loc[(df.year == df_strat_day.year) & (df.weekofyear == weekofyear_(df_strat_day))]
            if len(new_df_1) == 0:
                print(df_strat_day, "맨처음 시작일이 시작 요일(목)이 없을 경우")
                df_strat_day = df_strat_day - timedelta(weeks=1)
            elif len(new_df_1) >= 1:
                break
        ww = start_dayofweek
        while True:

            new_df = new_df_1.loc[(new_df_1.day_of_week == ww)]
            if (len(new_df) == 0):
                ww = ww - 1
            if len(new_df) == 1:
                break
            if ww == -1:
                break  # 이부분은 이전주로 가야함.

        if (ww == -1) and len(new_df_1) >= 1:
            #             print("지울것. 다른요일만 남은날")
            wk = 6
            while True:
                new_df = new_df_1.loc[(new_df_1.day_of_week == wk)]
                if (len(new_df) == 0):
                    wk = wk - 1
                if len(new_df) == 1:
                    break
                if wk == -1:
                    break  # 이부분은 이전주로 가야함.

        if len(new_df_1) == 0:
            print(df_strat_day, "맨처음 볼 테이블이 없음")

        else:
            df_end_day = df_strat_day = new_df.time.iloc[0]
            df_end_day = df_end_day + timedelta(weeks=1)

            while True:
                new_df_1 = df.loc[(df.year == df_end_day.year) & (df.weekofyear == weekofyear_(df_end_day))]
                if len(new_df_1) == 0:
#                     print(df_end_day, "끝나는 요일이 시작 요일(금)이 없을 경우")
                    df_end_day = df_end_day + timedelta(weeks=1)

                if (df.iloc[len(df) - 1, :].time < df_end_day):

                    return info_dataFrame[1:].sort_values(by=["time"]).reset_index(drop=True)

                elif len(new_df_1) >= 1:
                    if (new_df_1.time.iloc[0] < df_strat_day):  # 해년도를 넘어간 경우
                        print("해년도를 넘어간 경우", i)
                        df_end_day = df_end_day + timedelta(weeks=1)
                        #                         new_df_1 = df.loc[(df.year == df_end_day.year) & (df.weekofyear == df_end_day.weekofyear)&(df.time < df_end_day)]
                        new_df_1 = df.loc[(df.year == df_end_day.year) & (df.weekofyear == weekofyear_(df_end_day))]
                    break

            wk = end_dayofweek
            while True:  # 다음다음다음 날들을 찾음. 다 없으면 종료..
                new_df = new_df_1.loc[(new_df_1.day_of_week == wk)]
                if (len(new_df) == 0):
                    wk = wk + 1
                if len(new_df) == 1:
                    break
                if wk == 7:
                    #                     if (new_df_1.time.iloc[0] < df_strat_day): # 해년도를 넘어간 경우
                    #                         print("해년도를 넘어간 경우",i)
                    #                         df_end_day = df_end_day + timedelta(weeks = 1)
                    #                         new_df_1 = df.loc[(df.year == df_end_day.year) & (df.weekofyear == df_end_day.weekofyear)]
                    break  # 이부분은 이전주로 가야함.

            if (wk == 7) and len(new_df_1) >= 1:  #
                #     print("지울것. 금요일만 남은낧",i)
                while True:  # 1일 이상인 다음주를 찾음.
                    df_end_day = df_end_day + timedelta(weeks=1)
                    new_df_1 = df.loc[(df.year == df_end_day.year) & (df.weekofyear == weekofyear_(df_end_day))]
                    if len(new_df_1) >= 1:
                        break

                wk = 0
                while True:
                    new_df = new_df_1.loc[(new_df_1.day_of_week == wk)]
                    if (len(new_df) == 0):
                        wk = wk + 1
                    if len(new_df) == 1:
                        break
                #                 if (new_df_1.time.iloc[0] < df_strat_day): # 해년도를 넘어간 경우

                #                     print("해년도를 넘어간 경우",i)
                #                     df_end_day = df_end_day + timedelta(weeks = 1)
                #                     new_df_1 = df.loc[(df.year == df_end_day.year) & (df.weekofyear == df_end_day.weekofyear)]
                # elif
                # wk == 7:
                

        if len(new_df_1) == 0:
            print(df_end_day, "맨 마지막 볼 테이블이 없음",i)
        if len(new_df) == 0:
            print(df_end_day, "맨 마지막 볼 테이블이 없음",i)
        else:
            df_end_day = new_df.time.iloc[0]

            df_table = df.loc[(df.time.between(df_strat_day, df_end_day))]

#             if len(df_table) >= 6:
#                 print(i, "========",df_table)

            info = {"time": df_end_day, "오픈": df_table.iloc[0,].오픈, "저가": min(df_table.저가.values), "고가": max(df_table.고가.values), "종가": df_table[df_table.time == df_end_day].종가.iloc[0], "진입시점": df_strat_day}
            new_info_dataFrame = pd.DataFrame(info, index=[0])
            info_dataFrame = pd.concat([info_dataFrame, new_info_dataFrame], axis=0)  # 수익 데이터 저장


    return info_dataFrame[1:].sort_values(by=["time"]).reset_index(drop=True)

In [9]:
# 0월 1화 2수 3목 4금 5토 6일
info_dataFrame = out_put(df, start_dayofweek = 3, end_dayofweek = 2)
# info_dataFrame.to_csv("info_dataFrame.csv", encoding = "cp949")

 99%|██████████████████████████████████████████████████████████████████████████████▎| 222/224 [00:01<00:00, 185.04it/s]


In [10]:
info_dataFrame = info_dataFrame.rename(columns = {"진입시점":"시작일"})
info_dataFrame["timedelta"] = info_dataFrame["time"] - info_dataFrame["시작일"]
info_dataFrame['시작일'] = pd.to_datetime(info_dataFrame['시작일'], format='%Y-%m-%d', errors='raise')

In [11]:
for i in range(len(info_dataFrame)):
    df_table = df.loc[(df.time.between(info_dataFrame.iloc[i].시작일,info_dataFrame.iloc[i].time))]
    if len(df_table) >= 7:
        
        print(df_table)
        print(i,"========")

In [12]:
df2 = info_dataFrame[info_dataFrame["timedelta"]>timedelta(weeks = 1)]
df2  = df2.reset_index(drop = True)

In [13]:
for i in range(len(df2)):
    df_table = df.loc[(df.time.between(df2.iloc[i].시작일,df2.iloc[i].time))]
    
    print(df_table)
#     time.sleep(0.5)
    
    
    

         time      종가      오픈      고가      저가  day_of_year  weekofyear  year  \
8  2019-10-02  258.10  262.20  262.40  258.00          275          40  2019   
9  2019-10-04  256.90  257.40  260.20  256.90          277          40  2019   
10 2019-10-07  258.15  258.85  259.70  256.95          280          41  2019   
11 2019-10-08  261.70  258.30  261.95  258.15          281          41  2019   
12 2019-10-10  259.80  257.65  260.75  256.00          283          41  2019   

    day_of_week  year_of_month  
8             2             10  
9             4             10  
10            0             10  
11            1             10  
12            3             10  
         time      종가      오픈      고가      저가  day_of_year  weekofyear  year  \
68 2019-12-30  286.80  287.40  288.15  286.45          364          53  2019   
69 2020-01-02  282.35  286.25  287.15  282.35            2           1  2020   
70 2020-01-03  283.10  285.35  287.15  281.25            3           1  2020   
7

### 여기서부터 기술적 지표

In [18]:
WINDOW  = 3 # 몇 단튀(ex일)로 할것인지.

# 단순 이동 평균(Simple Moving Average, SMA)
def SMA(data, period=30, column='Close'):
    return data[column].rolling(window=period).mean()

# 지수 이동 평균(Exponential Moving Average, EMA)
def EMA(data, period=3, column='Close'):
    return data[column].ewm(span=period, adjust=False).mean()

def WMA(data, period=3, column='Close'): #가중이동평균
    val = []
    for i in range(WINDOW):
        val.append(None)

    for i in range(len(data)-period):
        df = data[[column]].loc[i+1:i+period].reset_index(drop = True)
        sum_count_1 = 0
        sum_count_2 = 0
        for j in range(period):
            sum_count_1 = sum_count_1 + (j+1) * float(df.iloc[j].values)
            sum_count_2 = sum_count_2 + (j+1)
        val.append(sum_count_1/sum_count_2)
    return val

def LSMA(data, period=3, column='Close'): # 최소제솝평균..

    val = []
    for i in range(WINDOW):
        val.append(None)

    for i in range(len(data)-period):
        df = data[[column]].loc[i+1:i+period].reset_index(drop = True)
        X = [k+1 for k in range(period)]
        Y = [float(df.iloc[k]) for k in range(period)]
        LRS = (len(X) * sum([X[i]*Y[i] for i in range(len(X))])- sum(X)*sum(Y)) /(len(X) * sum([X[i]*X[i] for i in range(len(X))]) - sum(X) * sum(X))
        LRT = (sum(Y) - LRS * sum(X)) / (len(X))
        LRI = LRT + LRS * len(X)
        val.append(LRI)

    return val

# data, period, column = data, WINDOW, '종가'
def triangular_weighted_moving_average(data, period=20, column='Close'):
    
    val = []
    for i in range(WINDOW):
        val.append(None)

    for i in range(len(data)-period):
        df = data[[column]].loc[i+1:i+period].reset_index(drop = True)
        
        A = int(np.ceil((period+0.1)/2)) # N기간을 2로나눔 (소수점 반올림)
        B = float(df.iloc[0:A].mean()) #A기간동안의 이동평균값
        for_Trima = list(df[column].values)
        for_Trima.insert(0, B)
        Trima = np.mean(for_Trima)#B값의 A기간 이동평균값
        val.append(np.mean(Trima))
        
    return val


In [19]:
data = info_dataFrame.reset_index(drop = True).copy()

In [20]:
del data["timedelta"]

In [21]:
# data[["단순이동평균_"+str(WINDOW)]] = SMA(data, period=WINDOW, column='종가')
# data[["지수이동평균_"+str(WINDOW)]] = EMA(data, period=WINDOW, column='종가')
# data[["가중이동평균_"+str(WINDOW)]] = WMA(data, period=WINDOW, column='종가')
# data[["최소제곱평균_"+str(WINDOW)]] = LSMA(data, period=WINDOW, column='종가')
# data[["삼각이동평균_"+str(WINDOW)]] = triangular_weighted_moving_average(data, period=WINDOW, column='종가')


In [22]:
# data["단순이동평균_"+"시가"] = SMA(data, period=WINDOW, column='오픈')
# data["지수이동평균_"+"시가"] = EMA(data, period=WINDOW, column='오픈')
# data["가중이동평균_"+"시가"] = WMA(data, period=WINDOW, column='오픈')
# data["최소제곱평균_"+"시가"] = LSMA(data, period=WINDOW, column='오픈')
# data["삼각이동평균_"+"시가"] = triangular_weighted_moving_average(data, period=WINDOW, column='오픈')


In [23]:
data["단순이동평균_"+"종가"] = SMA(data, period=WINDOW, column='종가')
data["지수이동평균_"+"종가"] = EMA(data, period=WINDOW, column='종가')
data["가중이동평균_"+"종가"] = WMA(data, period=WINDOW, column='종가')
data["최소제곱평균_"+"종가"] = LSMA(data, period=WINDOW, column='종가')
data["삼각이동평균_"+"종가"] = triangular_weighted_moving_average(data, period=WINDOW, column='종가')


In [24]:
unname_list = ["단순이동평균_","지수이동평균_","가중이동평균_","최소제곱평균_","삼각이동평균_"]

In [25]:
val_col_list = set(data.columns)

In [26]:

val_col_list = set(data.columns) - {"시가","고가","저가","종가","날짜","오픈","time","진입시점","시작일"}
val_col_list = set(val_col_list)
val_col_list = list(val_col_list)

data['time'] = pd.to_datetime(data['time'], format='%Y-%m-%d', errors='raise')
data = data.fillna(0)



In [134]:
for l in unname_list:

    name = [s for s in val_col_list if l in s]
    if (name[0].find("종가")):
        data["종가"+"-"+str(name[0])] = data["종가"] - data[name[0]]
        data.loc[list(data.loc[(data[name[0]]==0)][name[0]].index),"종가"+"-"+str(name[0])] = 0 # 원래 0 이였던 값들을 찾은다음 다시 0 을 집어넣어줌.
        data.loc[(data["종가"+"-"+str(name[0])])> 0,l+"포지션"] = "상방"
        data.loc[(data["종가"+"-"+str(name[0])])< 0,l+"포지션"] = "하방"

In [135]:
data.to_csv(save_path+"ALL_DATA_KOSPI_future_목시수종.csv", encoding = "cp949")

In [106]:
data

,time,오픈,저가,고가,종가,시작일,단순이동평균_종가,지수이동평균_종가,가중이동평균_종가,최소제곱평균_종가,...,종가-단순이동평균_종가,단순이동평균_포지션,종가-지수이동평균_종가,지수이동평균_포지션,종가-가중이동평균_종가,가중이동평균_포지션,종가-최소제곱평균_종가,최소제곱평균_포지션,종가-삼각이동평균_종가,삼각이동평균_포지션
0,2019-09-25,264.35,263.80,267.85,264.00,2019-09-20,0.000000,264.000000,0.000000,0.000000,...,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.00000,NaN
1,2019-10-02,264.90,258.00,266.00,258.10,2019-09-26,0.000000,261.050000,0.000000,0.000000,...,0.000000,NaN,-2.950000,하방,0.000000,NaN,0.000000,NaN,0.00000,NaN
2,2019-10-10,262.20,256.00,262.40,259.80,2019-10-02,260.633333,260.425000,0.000000,0.000000,...,-0.833333,하방,-0.625000,하방,0.000000,NaN,0.000000,NaN,0.00000,NaN
3,2019-10-16,257.65,256.00,267.60,266.40,2019-10-10,261.433333,263.412500,262.816667,265.583333,...,4.966667,상방,2.987500,상방,3.583333,상방,0.816667,상방,5.58750,상방
4,2019-10-23,266.80,262.40,268.15,266.10,2019-10-17,264.100000,264.756250,265.150000,267.250000,...,2.000000,상방,1.343750,상방,0.950000,상방,-1.150000,하방,2.25000,상방
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,2023-11-15,326.40,322.60,336.40,335.70,2023-11-09,324.250000,328.637117,328.283333,336.350000,...,11.450000,상방,7.062883,상방,7.416667,상방,-0.650000,하방,12.88125,상방
218,2023-11-22,334.80,331.95,339.00,338.45,2023-11-16,333.233333,333.543559,335.383333,339.683333,...,5.216667,상방,4.906441,상방,3.066667,상방,-1.233333,하방,5.86875,상방
219,2023-11-29,337.90,333.75,339.20,338.00,2023-11-23,337.383333,335.771779,337.766667,338.533333,...,0.616667,상방,2.228221,상방,0.233333,상방,-0.533333,하방,0.69375,상방
220,2023-12-06,337.50,333.10,339.65,333.95,2023-11-30,336.800000,334.860890,336.050000,334.550000,...,-2.850000,하방,-0.910890,하방,-2.100000,하방,-0.600000,하방,-3.20625,하방


In [107]:
# for l in unname_list:

#     name = [s for s in val_col_list if l in s]
# #     name = list(set(name))
# #     print(name)
#     if (name[0].find("시가") == -1):
#         data[name[1]+"-"+name[0]] = data[name[1]] - data[name[0]]

#         data.loc[(data[name[1]+"-"+name[0]]>0),l+"포지션"] = "상방"
#         data.loc[(data[name[1]+"-"+name[0]]<0),l+"포지션"] = "하방"
#     else:
#         data[name[0]+"-"+name[1]] = data[name[0]] - data[name[1]]

#         data.loc[(data[name[0]+"-"+name[1]]>0),l+"포지션"] = "상방"
#         data.loc[(data[name[0]+"-"+name[1]]<0),l+"포지션"] = "하방"
# data.to_csv("ALL_DATA_KOSPI_future.csv", encoding = "cp949")

In [108]:
data.columns

Index(['time', '오픈', '저가', '고가', '종가', '시작일', '단순이동평균_종가', '지수이동평균_종가',
       '가중이동평균_종가', '최소제곱평균_종가', '삼각이동평균_종가', '종가-단순이동평균_종가', '단순이동평균_포지션',
       '종가-지수이동평균_종가', '지수이동평균_포지션', '종가-가중이동평균_종가', '가중이동평균_포지션',
       '종가-최소제곱평균_종가', '최소제곱평균_포지션', '종가-삼각이동평균_종가', '삼각이동평균_포지션'],
      dtype='object')

In [109]:
# data.to_csv("info_dataFrame.csv", encoding = "cp949")

In [110]:
info_2 = {}
for j in tqdm(val_col_list):
    col_name = j
    
    info_2[col_name] = []
    
    for i in range(len(data)):
        if(data.iloc[i,:][col_name] == "None"):
            info_2[col_name].append(None)
        else:
            new_df = df.loc[df.time.between(data.iloc[i,:].시작일,data.iloc[i,:].time)]

            if (new_df.iloc[1,:].오픈 <= data.iloc[i,:][col_name]):
                state = "Long"
            else: # new_df.iloc[1,:].오픈 >= data.iloc[i,:][col_name])
                state = "Short"
#             print(state)

            if state == "Long":
                info_2[col_name].append(data.iloc[i,:][col_name] <= max(new_df.고가))
#                 print(data.iloc[i,:][col_name] , max(new_df.고가))

            elif state == "Short":
                info_2[col_name].append(data.iloc[i,:][col_name] >= min(new_df.저가))
#                 print(data.iloc[i,:][col_name] , min(new_df.저가))

info_2 = pd.DataFrame(info_2)
for j in val_col_list:
    print(j)
    print("총 날짜",len(info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))
    print("맞은 날짜",sum((info_2[(info_2[j] == True) |  (info_2[j] == False)][j])))
    print("정답률",sum((info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))/len(info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))
    print("=====================================================================")

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  3.10it/s]

단순이동평균_종가
총 날짜 222
맞은 날짜 190
정답률 0.8558558558558559
가중이동평균_종가
총 날짜 222
맞은 날짜 206
정답률 0.9279279279279279
지수이동평균_종가
총 날짜 222
맞은 날짜 202
정답률 0.9099099099099099
삼각이동평균_종가
총 날짜 222
맞은 날짜 178
정답률 0.8018018018018018
최소제곱평균_종가
총 날짜 222
맞은 날짜 206
정답률 0.9279279279279279


In [111]:
info_2.to_csv("info_dataFrame2_futrue.csv", encoding = "cp949")

In [182]:
data.loc[(data["time"] == "2019-09-25")]

,time,오픈,저가,고가,종가,시작일,단순이동평균_종가,지수이동평균_종가,가중이동평균_종가,최소제곱평균_종가,...,종가-단순이동평균_종가,단순이동평균_포지션,종가-지수이동평균_종가,지수이동평균_포지션,종가-가중이동평균_종가,가중이동평균_포지션,종가-최소제곱평균_종가,최소제곱평균_포지션,종가-삼각이동평균_종가,삼각이동평균_포지션


In [183]:
data

,time,오픈,저가,고가,종가,시작일,단순이동평균_종가,지수이동평균_종가,가중이동평균_종가,최소제곱평균_종가,...,종가-단순이동평균_종가,단순이동평균_포지션,종가-지수이동평균_종가,지수이동평균_포지션,종가-가중이동평균_종가,가중이동평균_포지션,종가-최소제곱평균_종가,최소제곱평균_포지션,종가-삼각이동평균_종가,삼각이동평균_포지션
0,2019-10-02,275.45,268.55,276.55,268.65,2019-09-26,0.000000,268.650000,0.000000,0.000000,...,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.00000,NaN
1,2019-10-10,272.75,266.55,272.95,270.35,2019-10-02,0.000000,269.500000,0.000000,0.000000,...,0.000000,NaN,0.850000,상방,0.000000,NaN,0.000000,NaN,0.00000,NaN
2,2019-10-16,268.20,266.55,278.15,276.95,2019-10-10,271.983333,273.225000,0.000000,0.000000,...,4.966667,상방,3.725000,상방,0.000000,NaN,0.000000,NaN,0.00000,NaN
3,2019-10-23,277.35,272.95,278.70,276.65,2019-10-17,274.650000,274.937500,275.700000,277.800000,...,2.000000,상방,1.712500,상방,0.950000,상방,-1.150000,하방,2.25000,상방
4,2019-10-30,277.80,274.90,279.95,275.70,2019-10-24,276.433333,275.318750,276.225000,275.808333,...,-0.733333,하방,0.381250,상방,-0.525000,하방,-0.108333,하방,-0.82500,하방
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215,2023-11-08,316.05,315.25,335.30,325.10,2023-11-02,318.000000,321.115055,319.208333,321.625000,...,7.100000,상방,3.984945,상방,5.891667,상방,3.475000,상방,7.98750,상방
216,2023-11-15,325.95,322.15,335.95,335.25,2023-11-09,323.800000,328.182527,327.833333,335.900000,...,11.450000,상방,7.067473,상방,7.416667,상방,-0.650000,하방,12.88125,상방
217,2023-11-22,334.35,331.50,338.55,338.00,2023-11-16,332.783333,333.091264,334.933333,339.233333,...,5.216667,상방,4.908736,상방,3.066667,상방,-1.233333,하방,5.86875,상방
218,2023-11-29,337.45,333.30,338.75,337.55,2023-11-23,336.933333,335.320632,337.316667,338.083333,...,0.616667,상방,2.229368,상방,0.233333,상방,-0.533333,하방,0.69375,상방


In [877]:
aa = 0#진입시점
info_2 = {}
for j in tqdm(val_col_list):
    col_name = j
    
    info_2[col_name] = []
    
    for i in range(len(data)):
        if(data.iloc[i,:][col_name] == "None"):
            info_2[col_name].append(None)
        else:
            new_df = df.loc[df.time.between(data.iloc[i,:].진입시점,data.iloc[i,:].time)]

            if (new_df.iloc[aa,:].오픈 <= data.iloc[i,:][col_name]):
                state = "Long"
            else: # new_df.iloc[1,:].오픈 >= data.iloc[i,:][col_name])
                state = "Short"
#             print(state)

            if state == "Long":
                info_2[col_name].append(data.iloc[i,:][col_name] <= max(new_df.고가))
#                 print(data.iloc[i,:][col_name] , max(new_df.고가))

            elif state == "Short":
                info_2[col_name].append(data.iloc[i,:][col_name] >= min(new_df.저가))
#                 print(data.iloc[i,:][col_name] , min(new_df.저가))

info_2 = pd.DataFrame(info_2)
for j in val_col_list:
    print(j)
    print("총 날짜",len(info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))
    print("맞은 날짜",sum((info_2[(info_2[j] == True) |  (info_2[j] == False)][j])))
    print("정답률",sum((info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))/len(info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))
    print("=====================================================================")

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:11<00:00,  2.25s/it]

가중이동평균_3
총 날짜 1124
맞은 날짜 716
정답률 0.6370106761565836
삼각이동평균_3
총 날짜 1124
맞은 날짜 567
정답률 0.5044483985765125
최소제곱평균_3
총 날짜 1124
맞은 날짜 934
정답률 0.8309608540925267
지수이동평균_3
총 날짜 1127
맞은 날짜 683
정답률 0.6060337178349601
단순이동평균_3
총 날짜 1125
맞은 날짜 600
정답률 0.5333333333333333


In [805]:
# 진입 시점대비..
info_2 = {}
for j in tqdm(val_col_list):
    col_name = j
    
    info_2[col_name] = []
    
    for i in range(len(data)):
        if(data.iloc[i,:][col_name] == "None"):
            info_2[col_name].append(None)
        else:
            new_df = df.loc[df.time.between(data.iloc[i,:].진입시점,data.iloc[i,:].time)]

            if (new_df.iloc[1,:].오픈 <= data.iloc[i,:][col_name]):
                state = "Long"
            else: # new_df.iloc[1,:].오픈 >= data.iloc[i,:][col_name])
                state = "Short"
#             print(state)

            if state == "Long":
                info_2[col_name].append(new_df.iloc[1,:].오픈 <= max(new_df.고가))
#                 print(data.iloc[i,:][col_name] , max(new_df.고가))

            elif state == "Short":
                info_2[col_name].append(new_df.iloc[1,:].오픈 >= min(new_df.저가))
#                 print(data.iloc[i,:][col_name] , min(new_df.저가))

info_2 = pd.DataFrame(info_2)
for j in val_col_list:
    print(j)
    print("총 날짜",len(info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))
    print("맞은 날짜",sum((info_2[(info_2[j] == True) |  (info_2[j] == False)][j])))
    print("정답률",sum((info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))/len(info_2[(info_2[j] == True) |  (info_2[j] == False)][j]))
    print("=====================================================================")

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:11<00:00,  2.24s/it]

가중이동평균_3
총 날짜 1124
맞은 날짜 1124
정답률 1.0
삼각이동평균_3
총 날짜 1124
맞은 날짜 1124
정답률 1.0
최소제곱평균_3
총 날짜 1124
맞은 날짜 1124
정답률 1.0
지수이동평균_3
총 날짜 1127
맞은 날짜 1127
정답률 1.0
단순이동평균_3
총 날짜 1125
맞은 날짜 1125
정답률 1.0


False